import necessary libraries

In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

load dataset 

In [2]:
df = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\My_Python\Project\dataset\Car_details_v3_CLEANED.csv") #dataset path location 
X = df.drop("selling_price", axis=1)
y = df["selling_price"]

separate numerical and categorical features

In [3]:
numeric_features = ["year", "km_driven", "engine","max_power", "seats","owner", "mileage_value"]
categorical_features = ["fuel", "seller_type","transmission", "mileage_unit"]

preprocessing

In [4]:
numeric_transformer = Pipeline([("scaler", StandardScaler())])
categorical_transformer = Pipeline([("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num", numeric_transformer, numeric_features),("cat", categorical_transformer, categorical_features)])

split into train and test

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models used

In [6]:
models = {
    "KNN": KNeighborsRegressor(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=42)
}

parameters for hyperparameter tuning of models

In [7]:
model_configs = {
    "KNN": {
        "model": KNeighborsRegressor(),
        "params": {
            "model__n_neighbors": [3, 5, 7, 9],
            "model__weights": ["uniform", "distance"]
        }
    },
    "DecisionTree": {
        "model": DecisionTreeRegressor(random_state=42),
        "params": {
            "model__max_depth": [None, 10, 20, 30],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4]
        }
    },
    "RandomForest": {
        "model": RandomForestRegressor(random_state=42, n_jobs=-1),
        "params": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_split": [2, 5],
            "model__min_samples_leaf": [1, 2]
        }
    },
    "GradientBoosting": {
        "model": GradientBoostingRegressor(random_state=42),
        "params": {
            "model__n_estimators": [200, 300, 500],
            "model__learning_rate": [0.05, 0.1],
            "model__max_depth": [3, 4, 5],
            "model__min_samples_split": [2, 5],
            "model__min_samples_leaf": [1, 2],
            "model__subsample": [0.8, 1.0]
        }
    }
}

hyperparameter tuning models

In [8]:
trained_models = {}
model_scores = {}

for name, config in model_configs.items():
    print(f"\nTuning {name}...")

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", config["model"])
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=config["params"],
        cv=5,
        scoring="r2",
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    trained_models[name] = best_model
    model_scores[name] = r2

    print(f"{name} best R2: {r2:.4f}")
    print(f"{name} best params: {grid.best_params_}")


Tuning KNN...
Fitting 5 folds for each of 8 candidates, totalling 40 fits
KNN best R2: 0.8559
KNN best params: {'model__n_neighbors': 3, 'model__weights': 'distance'}

Tuning DecisionTree...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
DecisionTree best R2: 0.8843
DecisionTree best params: {'model__max_depth': None, 'model__min_samples_leaf': 4, 'model__min_samples_split': 10}

Tuning RandomForest...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
RandomForest best R2: 0.9169
RandomForest best params: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}

Tuning GradientBoosting...
Fitting 5 folds for each of 144 candidates, totalling 720 fits
GradientBoosting best R2: 0.9209
GradientBoosting best params: {'model__learning_rate': 0.1, 'model__max_depth': 4, 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 500, 'model__subsample': 0.8}


select best model

In [9]:
best_model_name = max(model_scores, key=model_scores.get)
best_model = trained_models[best_model_name]
print(f"Best Model: {best_model_name}")
print(f"Best R2 Score: {model_scores[best_model_name]:.4f}")

Best Model: GradientBoosting
Best R2 Score: 0.9209


Save models using pickle

In [12]:
import joblib

joblib.dump(trained_models, "all_models.pkl", compress=9)
joblib.dump(model_scores, "model_scores.pkl", compress=9)
joblib.dump(best_model, "best_model.pkl", compress=9)

print("Models saved successfully")

Models saved successfully
